# Live Stream Demo — Facial Emotion Recognition (FER2013)
Real-time emotion recognition from the **webcam**, using the **best model** of the project (Full Fine-Tuning, EfficientNet-B0) exported to **ONNX FP16**.
No training needed: the model is downloaded from the project's GitHub repository. Works on the free CPU runtime.

**How to use:** `Runtime → Run all`, then open the public link printed at the end, allow the camera, and press **Record**.

<div dir="rtl">

# عرض البث المباشر — التعرّف على مشاعر الوجه
تعرّف على المشاعر لحظياً من **الكاميرا** باستخدام **أفضل نموذج** في المشروع (الضبط الدقيق الكامل) بعد تحويله إلى **ONNX FP16**.
لا يحتاج تدريباً: النموذج يُنزَّل من مستودع المشروع على GitHub، ويعمل على المعالج العادي.

**طريقة الاستخدام:** `Runtime ← Run all`، ثم افتحي الرابط العام الذي يظهر في الأخير، واسمحي بالكاميرا، واضغطي **Record**.

</div>

In [ ]:
!pip install -q onnxruntime gradio "opencv-python-headless<5"

### 1. Download the best model (ONNX FP16, 8 MB)
<div dir="rtl">

### 1. تنزيل أفضل نموذج (ONNX FP16، حجمه 8 ميجا)

</div>

In [ ]:
import urllib.request, os
MODEL_URL = "https://github.com/amanialanazi/FER2013-Emotion-Recognition/raw/main/models/fer_efficientnet_b0_fp16.onnx"
MODEL_PATH = "fer_efficientnet_b0_fp16.onnx"
if not os.path.exists(MODEL_PATH):
    try:
        urllib.request.urlretrieve(MODEL_URL, MODEL_PATH)
    except Exception as e:   # no internet access to GitHub -> upload the file by hand
        print("Download failed:", e, "-> please upload fer_efficientnet_b0_fp16.onnx")
        from google.colab import files
        files.upload()
print("Model:", MODEL_PATH, f"{os.path.getsize(MODEL_PATH) / 1e6:.1f} MB")

### 2. The prediction pipeline (same preprocessing as training)
`frame → detect faces (OpenCV) → crop → grayscale → resize 224 → normalize → ONNX model → emotion + confidence → draw on the frame`
<div dir="rtl">

### 2. مسار التوقّع (نفس معالجة التدريب)
كل إطار من الفيديو: اكتشاف الوجوه ← قص الوجه ← رمادي ← تكبير إلى 224 ← تطبيع ← النموذج ← الشعور ونسبة الثقة ← الرسم على الإطار.

</div>

In [ ]:
import time
import cv2
import numpy as np
import onnxruntime as ort

CLASS_NAMES = ["angry", "disgust", "fear", "happy", "neutral", "sad", "surprise"]
COLORS = {"angry": (230, 60, 60), "disgust": (60, 160, 60), "fear": (160, 80, 180), "happy": (240, 200, 40),
          "neutral": (200, 200, 200), "sad": (60, 140, 220), "surprise": (255, 160, 40)}   # RGB
MEAN = np.array([0.485, 0.456, 0.406], dtype=np.float32).reshape(3, 1, 1)   # ImageNet mean
STD = np.array([0.229, 0.224, 0.225], dtype=np.float32).reshape(3, 1, 1)    # ImageNet std

session = ort.InferenceSession(MODEL_PATH, providers=["CPUExecutionProvider"])
detector = cv2.CascadeClassifier(cv2.data.haarcascades + "haarcascade_frontalface_default.xml")

def preprocess(gray_face):
    face = cv2.resize(gray_face, (224, 224))
    x = face.astype(np.float32) / 255.0
    x = np.stack([x, x, x])                                  # 1 gray channel -> 3 channels
    return ((x - MEAN) / STD)[None].astype(np.float32)       # [1, 3, 224, 224]

def softmax(z):
    e = np.exp(z - z.max())
    return e / e.sum()

def annotate(frame_rgb):
    """Detect every face in one frame, predict its emotion and draw the result."""
    if frame_rgb is None:
        return None
    t0 = time.time()
    frame = frame_rgb.copy()
    gray = cv2.cvtColor(frame, cv2.COLOR_RGB2GRAY)
    faces = detector.detectMultiScale(gray, scaleFactor=1.1, minNeighbors=6, minSize=(60, 60))
    for (x, y, w, h) in faces:
        m = int(0.1 * w)
        x0, y0 = max(x - m, 0), max(y - m, 0)
        x1, y1 = min(x + w + m, gray.shape[1]), min(y + h + m, gray.shape[0])
        probs = softmax(session.run(None, {"image": preprocess(gray[y0:y1, x0:x1])})[0][0].astype(np.float32))
        label = CLASS_NAMES[int(probs.argmax())]
        cv2.rectangle(frame, (x0, y0), (x1, y1), COLORS[label], 3)
        cv2.putText(frame, f"{label} {100 * probs.max():.0f}%", (x0, max(y0 - 10, 25)),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.9, COLORS[label], 2)
    cv2.putText(frame, f"{len(faces)} face(s) | {1000 * (time.time() - t0):.0f} ms", (10, 25),
                cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 255, 0), 2)
    return frame

# Quick self-test on a blank frame / اختبار سريع
print("Pipeline ready:", annotate(np.zeros((240, 320, 3), dtype=np.uint8)).shape)

### 3. Live stream from the webcam (Gradio)
Open the **public URL** below (works on a laptop or a phone), allow the camera, press **Record**. Every ~0.2 s a frame is sent to the model and the annotated frame comes back.
<div dir="rtl">

### 3. البث المباشر من الكاميرا (Gradio)
افتحي **الرابط العام** الذي يظهر تحت (يعمل على اللابتوب والجوال)، واسمحي بالكاميرا، ثم اضغطي **Record**. كل 0.2 ثانية تقريباً يُرسل إطار للنموذج ويرجع وعليه النتيجة.

</div>

In [ ]:
import gradio as gr

with gr.Blocks(title="Live Facial Emotion Recognition") as demo:
    gr.Markdown("## Live Facial Emotion Recognition — EfficientNet-B0 (ONNX FP16)\nAllow the camera and press **Record**.")
    with gr.Row():
        cam = gr.Image(sources=["webcam"], streaming=True, type="numpy", label="Webcam")
        out = gr.Image(type="numpy", label="Live prediction")
    cam.stream(annotate, inputs=cam, outputs=out, stream_every=0.2, time_limit=600)

demo.launch(share=True, debug=False)